In [1]:
! git clone https://github.com/rashibharti28/BERT-Quantization-PTQ-QAT-on-dair-ai-emotion.git

Cloning into 'BERT-Quantization-PTQ-QAT-on-dair-ai-emotion'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 37 (delta 10), reused 12 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 355.13 KiB | 6.58 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Filtering content: 100% (2/2), 387.73 MiB | 21.12 MiB/s, done.


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
import evaluate
import numpy as np
import pandas as pd
from transformers import DataCollatorWithPadding
from datasets import load_dataset, DatasetDict, Dataset

In [ ]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
ds = load_dataset("dair-ai/emotion", "split")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:

train = ds['train']
val = ds['validation']
test = ds['test']

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [ ]:
id2labels = {0: 'sadness',
             1: 'joy',
             2: 'love',
             3: 'anger',
             4: 'fear',
             5: 'surprise'}
labels2id = {
        'sadness': 0,
        'joy' : 1,
        'love': 2,
        'anger':3,
        'fear' : 4,
        'surprise' : 5
    }

In [ ]:
l = {
    'i2l': id2labels,
    'l2i': labels2id
}
print(l)

{'i2l': {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}, 'l2i': {'sadness': 0, 'joy': 1, 'love': 2, 'anger': 3, 'fear': 4, 'surprise': 5}}


In [ ]:
model_path = "distilbert/distilbert-base-uncased"
finetune_model_dir = "/content/BERT-Quantization-PTQ-QAT-on-dair-ai-emotion/finetuning_bert"
tokenizer = AutoTokenizer.from_pretrained(finetune_model_dir)
model = AutoModelForSequenceClassification.from_pretrained(finetune_model_dir, num_labels = 6, id2label=l['i2l'], label2id = l['l2i'],)


In [ ]:
print(model.config.id2label)
print(model.config.label2id)


{0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}
{'sadness': 0, 'joy': 1, 'love': 2, 'anger': 3, 'fear': 4, 'surprise': 5}


In [ ]:
max_length = 128
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding='max_length', max_length=max_length)
# map datasets
tokenized_train = train.map(lambda x: tokenize_batch(x), batched=True)
tokenized_valid = val.map(lambda x: tokenize_batch(x), batched=True)
tokenized_test  = test.map(lambda x: tokenize_batch(x), batched=True)


# Set format for PyTorch compatibility
cols = ["input_ids", "attention_mask", "label"]
tokenized_train.set_format(type="torch", columns=cols)
tokenized_valid.set_format(type="torch", columns=cols)
tokenized_test.set_format(type="torch", columns=cols)



Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
def print_model_params(model):

    for name, param in model.named_parameters():
        print(f"{name:<60}  shape={tuple(param.shape)}  dtype={param.dtype}")

In [ ]:
print_model_params(model)

distilbert.embeddings.word_embeddings.weight                  shape=(30522, 768)  dtype=torch.float32
distilbert.embeddings.position_embeddings.weight              shape=(512, 768)  dtype=torch.float32
distilbert.embeddings.LayerNorm.weight                        shape=(768,)  dtype=torch.float32
distilbert.embeddings.LayerNorm.bias                          shape=(768,)  dtype=torch.float32
distilbert.transformer.layer.0.attention.q_lin.weight         shape=(768, 768)  dtype=torch.float32
distilbert.transformer.layer.0.attention.q_lin.bias           shape=(768,)  dtype=torch.float32
distilbert.transformer.layer.0.attention.k_lin.weight         shape=(768, 768)  dtype=torch.float32
distilbert.transformer.layer.0.attention.k_lin.bias           shape=(768,)  dtype=torch.float32
distilbert.transformer.layer.0.attention.v_lin.weight         shape=(768, 768)  dtype=torch.float32
distilbert.transformer.layer.0.attention.v_lin.bias           shape=(768,)  dtype=torch.float32
distilbert.transfo

In [ ]:
model.eval()


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
model_on_cpu = model.to("cpu")


In [ ]:
quantized_model_dynamic = torch.quantization.quantize_dynamic(
    model_on_cpu,
    {torch.nn.Linear},
    dtype=torch.qint8
)
quantized_model_dynamic.to("cpu")

/tmp/ipython-input-3396547684.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model_dynamic = torch.quantization.quantize_dynamic(


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (k_lin): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (v_lin): DynamicQuantizedLinear(in_features=768, out_features=768, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
            (out_lin): DynamicQuantizedLinear(in_features=768, out_featur

In [ ]:
from google.colab import drive
import os
import torch

# Mount Google Drive
drive.mount('/content/drive')

save_path = r"/content/drive/MyDrive/ptq"
os.makedirs(save_path, exist_ok=True)

torch.save(quantized_model_dynamic, os.path.join(save_path, "quantized_model.pt"))
tokenizer.save_pretrained(save_path)

print(f"Model and tokenizer saved at {save_path}")


Mounted at /content/drive
Model and tokenizer saved at /content/drive/MyDrive/ptq
